In [5]:
"""
Chile Mineral Supply Chain: Interactive Dashboard
==================================================
Generates a self-contained HTML dashboard with:
  1. Data source documentation
  2. Industry context (macroeconomic, production trends, exploration, sustainability)
  3. General inventory and data quality overview
  4. Supply chain visualization (map, production, exports)

Proper tripartite classification: active mines, idle mines, prospects.
Industry context drawn from GBR Chile Mining 2021 report and COCHILCO data.
Run after Chile_MainAn.ipynb (requires CSVs in Preliminary/).
Output: Chile/Outputs/Chile_Dashboard.html + Chile_Map_Layered.html
"""

import os
import json
import numpy as np
import pandas as pd
import folium
from folium import plugins
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio

# ── Paths ────────────────────────────────────────────────────────────────
BASE_DIR = "/Users/leoss/Desktop/GitHub/Capstone/Case studies/Chile"
PRELIM   = os.path.join(BASE_DIR, "Preliminary")
OUT_DIR  = os.path.join(BASE_DIR, "Outputs")
os.makedirs(OUT_DIR, exist_ok=True)

# ── Load data ────────────────────────────────────────────────────────────
inv     = pd.read_csv(os.path.join(PRELIM, "Chile_Minerals_Inventory.csv"), low_memory=False)
links   = pd.read_csv(os.path.join(PRELIM, "Chile_Mine_Plant_Links.csv"))
edges   = pd.read_csv(os.path.join(PRELIM, "Chile_Supply_Chain_Edges.csv"))
exports = pd.read_csv(os.path.join(PRELIM, "Chile_Export_Destinations.csv"))
ports   = pd.read_csv(os.path.join(PRELIM, "Chile_Ports.csv"))

print(f"Loaded: {len(inv)} facilities, {len(links)} links, {len(edges)} edges, {len(exports)} exports")

# ══════════════════════════════════════════════════════════════════════════
# 1. TRIPARTITE CLASSIFICATION
# ══════════════════════════════════════════════════════════════════════════

def classify_site(row):
    ft = str(row.get("FACILITY_TYPE", "")).strip()
    if ft in ("Mine (active)", "Mine (USGS)"):
        return "active"
    elif ft == "Mine (idle)":
        return "idle"
    elif ft == "Prospect/Project":
        return "prospect"
    else:
        return "processing"

inv["SITE_STATUS"] = inv.apply(classify_site, axis=1)

STAGE_MAP = {
    "Mine (active)": "extraction", "Mine (USGS)": "extraction",
    "Mine (idle)": "extraction_idle", "Prospect/Project": "prospect",
    "Concentrator": "concentration", "SX-EW Plant": "sx_ew",
    "Smelter": "smelting", "Refinery": "refining",
    "Processing Plant": "processing", "Pellet Plant": "processing",
    "Grinding Plant": "processing", "Steel Plant": "processing",
}
inv["CHAIN_STAGE_V2"] = inv["FACILITY_TYPE"].map(STAGE_MAP).fillna("other")

active_mines = inv[inv["SITE_STATUS"] == "active"].copy()
idle_mines   = inv[inv["SITE_STATUS"] == "idle"].copy()
prospects    = inv[inv["SITE_STATUS"] == "prospect"].copy()
processing   = inv[inv["SITE_STATUS"] == "processing"].copy()

print(f"Active: {len(active_mines)}, Idle: {len(idle_mines)}, "
      f"Prospects: {len(prospects)}, Processing: {len(processing)}")

# ══════════════════════════════════════════════════════════════════════════
# 2. COMPUTE SUMMARY STATISTICS
# ══════════════════════════════════════════════════════════════════════════

source_files = {
    "Chile_Minerals_Inventory.csv": inv,
    "Chile_Mine_Plant_Links.csv": links,
    "Chile_Supply_Chain_Edges.csv": edges,
    "Chile_Export_Destinations.csv": exports,
    "Chile_Ports.csv": ports,
}

source_rows = []
for fname, df in source_files.items():
    fpath = os.path.join(PRELIM, fname)
    fsize_kb = os.path.getsize(fpath) / 1024 if os.path.exists(fpath) else 0
    source_rows.append({"file": fname, "records": len(df), "columns": len(df.columns), "size_kb": fsize_kb})
source_df = pd.DataFrame(source_rows)

n_geocoded = inv[inv["LATITUD"].notna() & inv["LONGITUD"].notna()].shape[0]
pct_geocoded = n_geocoded / len(inv) * 100

comm_col = "PRIMARY_COMMODITY"
n_commodities = inv[comm_col].nunique() if comm_col in inv.columns else 0

res_cols = sorted([c for c in inv.columns if c.endswith(("_Resource", "_Reserve"))])
n_res_fields = len(res_cols)

cu_prod = inv[
    (inv["SITE_STATUS"] == "active") &
    inv["COCHILCO_CU_2024_KMT"].notna() &
    (inv["COCHILCO_CU_2024_KMT"] > 0)
].copy()
cu_prod = cu_prod.sort_values("COCHILCO_CU_2024_KMT", ascending=False).reset_index(drop=True)
cu_total = cu_prod["COCHILCO_CU_2024_KMT"].sum()
cu_prod["cumulative_pct"] = cu_prod["COCHILCO_CU_2024_KMT"].cumsum() / cu_total * 100
cu_prod["rank"] = range(1, len(cu_prod) + 1)

reg_col = next(
    (c for c in ["REGION", "NOMBRE_REGION", "ADM1"]
     if c in inv.columns and inv[c].notna().sum() > 10), None)
n_regions = inv[reg_col].nunique() if reg_col else 0
n_edge_types = edges["EDGE_TYPE"].nunique()

print(f"Cu total (active matched): {cu_total:,.0f} kMT from {len(cu_prod)} mines")

# ══════════════════════════════════════════════════════════════════════════
# 3. COLOR PALETTE
# ══════════════════════════════════════════════════════════════════════════

PAL = {
    "active": "#2c3e6b", "idle": "#a0a8b4", "prospect": "#d4940a",
    "conc": "#6c3483", "sx_ew": "#d4940a", "smelting": "#c0392b",
    "refining": "#e87461", "processing": "#1b7a6e", "port": "#16537e",
    "other": "#7f8c8d", "bg": "#f8f9fa", "text": "#2b2b2b", "text_sec": "#666666",
}

# ══════════════════════════════════════════════════════════════════════════
# 4. PLOTLY CHARTS: GENERAL DATA OVERVIEW
# ══════════════════════════════════════════════════════════════════════════

print("Building overview charts...")

# ── Chart A: Facility Type Breakdown ─────────────────────────────────────

ftype_counts = inv["FACILITY_TYPE"].value_counts()
ftype_color_map = {
    "Mine (active)": PAL["active"], "Mine (USGS)": "#3d5a8a",
    "Mine (idle)": PAL["idle"], "Prospect/Project": PAL["prospect"],
    "Concentrator": PAL["conc"], "SX-EW Plant": "#e6a817",
    "Smelter": PAL["smelting"], "Refinery": PAL["refining"],
    "Processing Plant": PAL["processing"],
}

figA = go.Figure()
figA.add_trace(go.Bar(
    y=ftype_counts.index[::-1], x=ftype_counts.values[::-1],
    orientation="h",
    marker_color=[ftype_color_map.get(ft, PAL["other"]) for ft in ftype_counts.index[::-1]],
    text=[f"{v}  ({v/len(inv)*100:.1f}%)" for v in ftype_counts.values[::-1]],
    textposition="outside", textfont=dict(size=10),
))
figA.update_layout(
    title=dict(text=f"All Facility Types<br><sub>{len(inv)} total records in inventory</sub>",
               font=dict(size=14)),
    xaxis_title="Count", height=400,
    margin=dict(l=160, r=80, t=70, b=40),
    plot_bgcolor="#f8f9fa", paper_bgcolor="white",
    xaxis=dict(gridcolor="#e0e0e0"),
    font=dict(family="Helvetica Neue, Arial, sans-serif", color="#2b2b2b"),
)
chartA_html = pio.to_html(figA, full_html=False, include_plotlyjs=False)

# ── Chart B: Site Status Breakdown ───────────────────────────────────────

status_summary = pd.DataFrame({
    "status": ["Active Mines", "Idle Mines", "Prospects", "Processing"],
    "count": [len(active_mines), len(idle_mines), len(prospects), len(processing)],
    "color": [PAL["active"], PAL["idle"], PAL["prospect"], PAL["processing"]],
})

figB = go.Figure()
figB.add_trace(go.Bar(
    x=status_summary["status"], y=status_summary["count"],
    marker_color=status_summary["color"],
    text=status_summary["count"], textposition="outside",
    textfont=dict(size=12, color="#2b2b2b"),
))
figB.update_layout(
    title=dict(text=f"Classified Status<br><sub>Tripartite extraction classification</sub>",
               font=dict(size=14)),
    yaxis_title="Count", height=380,
    margin=dict(l=60, r=40, t=80, b=50),
    plot_bgcolor="#f8f9fa", paper_bgcolor="white",
    yaxis=dict(gridcolor="#e0e0e0"),
    font=dict(family="Helvetica Neue, Arial, sans-serif", color="#2b2b2b"),
)
chartB_html = pio.to_html(figB, full_html=False, include_plotlyjs=False)

# ── Chart C: Commodity Profile ───────────────────────────────────────────

if comm_col in inv.columns:
    active_comms = active_mines[comm_col].value_counts().head(12)
    prospect_comms = prospects[comm_col].value_counts().head(12)
    idle_comms = idle_mines[comm_col].value_counts().head(12)
    all_comms_union = sorted(set(active_comms.index) | set(prospect_comms.index) | set(idle_comms.index))

    figC = go.Figure()
    figC.add_trace(go.Bar(name="Active", x=all_comms_union,
        y=[active_comms.get(c, 0) for c in all_comms_union], marker_color=PAL["active"]))
    figC.add_trace(go.Bar(name="Idle", x=all_comms_union,
        y=[idle_comms.get(c, 0) for c in all_comms_union], marker_color=PAL["idle"]))
    figC.add_trace(go.Bar(name="Prospect", x=all_comms_union,
        y=[prospect_comms.get(c, 0) for c in all_comms_union], marker_color=PAL["prospect"]))
    figC.update_layout(
        barmode="group",
        title=dict(text="Primary Commodity by Site Status", font=dict(size=14)),
        yaxis_title="Number of sites", height=420,
        margin=dict(l=60, r=40, t=60, b=100),
        plot_bgcolor="#f8f9fa", paper_bgcolor="white",
        yaxis=dict(gridcolor="#e0e0e0"),
        legend=dict(x=0.75, y=0.95),
        font=dict(family="Helvetica Neue, Arial, sans-serif", color="#2b2b2b"),
        xaxis=dict(tickangle=-40),
    )
    chartC_html = pio.to_html(figC, full_html=False, include_plotlyjs=False)
else:
    chartC_html = "<p>PRIMARY_COMMODITY column not found.</p>"

# ── Chart D: Resource/Reserve Coverage ───────────────────────────────────

extraction_all = inv[inv["SITE_STATUS"].isin(["active", "idle", "prospect"])].copy()
extraction_all["has_res_data"] = extraction_all[res_cols].notna().any(axis=1) if res_cols else False

cov_by_status = extraction_all.groupby("SITE_STATUS").agg(
    total=("has_res_data", "size"), has_data=("has_res_data", "sum"))
cov_by_status["missing"] = cov_by_status["total"] - cov_by_status["has_data"]
cov_by_status["pct"] = (cov_by_status["has_data"] / cov_by_status["total"] * 100).round(1)

status_display = {"active": "Active Mines", "idle": "Idle Mines", "prospect": "Prospects"}
figD = go.Figure()
figD.add_trace(go.Bar(name="Has resource/reserve data",
    x=[status_display.get(s, s) for s in cov_by_status.index],
    y=cov_by_status["has_data"], marker_color=PAL["processing"],
    text=[f"{p:.0f}%" for p in cov_by_status["pct"]], textposition="outside"))
figD.add_trace(go.Bar(name="Missing",
    x=[status_display.get(s, s) for s in cov_by_status.index],
    y=cov_by_status["missing"], marker_color="#e87461", opacity=0.6))
figD.update_layout(
    barmode="stack",
    title=dict(text="Resource/Reserve Data Coverage", font=dict(size=14)),
    yaxis_title="Number of sites", height=380,
    margin=dict(l=60, r=40, t=60, b=50),
    plot_bgcolor="#f8f9fa", paper_bgcolor="white",
    yaxis=dict(gridcolor="#e0e0e0"),
    legend=dict(x=0.55, y=0.95),
    font=dict(family="Helvetica Neue, Arial, sans-serif", color="#2b2b2b"),
)
chartD_html = pio.to_html(figD, full_html=False, include_plotlyjs=False)

# ── Chart E: Regional Distribution ───────────────────────────────────────

if reg_col:
    reg_by_status = inv[inv["SITE_STATUS"].isin(["active", "idle", "prospect"])].copy()
    reg_pivot = reg_by_status.groupby([reg_col, "SITE_STATUS"]).size().unstack(fill_value=0)
    reg_pivot["total"] = reg_pivot.sum(axis=1)
    reg_pivot = reg_pivot.sort_values("total", ascending=True)

    figE = go.Figure()
    for status, color in [("active", PAL["active"]), ("idle", PAL["idle"]), ("prospect", PAL["prospect"])]:
        if status in reg_pivot.columns:
            figE.add_trace(go.Bar(name=status_display.get(status, status),
                y=[str(r)[:30] for r in reg_pivot.index],
                x=reg_pivot[status], orientation="h", marker_color=color))
    figE.update_layout(
        barmode="stack",
        title=dict(text="Regional Distribution of Extraction Sites", font=dict(size=14)),
        xaxis_title="Number of sites", height=500,
        margin=dict(l=200, r=40, t=60, b=50),
        plot_bgcolor="#f8f9fa", paper_bgcolor="white",
        xaxis=dict(gridcolor="#e0e0e0"),
        legend=dict(x=0.7, y=0.05),
        font=dict(family="Helvetica Neue, Arial, sans-serif", color="#2b2b2b"),
    )
    chartE_html = pio.to_html(figE, full_html=False, include_plotlyjs=False)
else:
    chartE_html = "<p>No region column found in inventory.</p>"

# ── Chart F: Geocoding Coverage ──────────────────────────────────────────

geo_by_status = inv.groupby("SITE_STATUS").apply(
    lambda g: pd.Series({
        "geocoded": g["LATITUD"].notna().sum(),
        "missing": g["LATITUD"].isna().sum(),
    })
).reset_index()

figF = go.Figure()
figF.add_trace(go.Bar(name="Geocoded",
    x=[status_display.get(s, s) for s in geo_by_status["SITE_STATUS"]],
    y=geo_by_status["geocoded"], marker_color=PAL["processing"]))
figF.add_trace(go.Bar(name="Missing coordinates",
    x=[status_display.get(s, s) for s in geo_by_status["SITE_STATUS"]],
    y=geo_by_status["missing"], marker_color="#e87461", opacity=0.6))
figF.update_layout(
    barmode="stack",
    title=dict(text=f"Geocoding Coverage<br><sub>{n_geocoded}/{len(inv)} ({pct_geocoded:.1f}%) have coordinates</sub>",
               font=dict(size=14)),
    yaxis_title="Number of sites", height=380,
    margin=dict(l=60, r=40, t=80, b=50),
    plot_bgcolor="#f8f9fa", paper_bgcolor="white",
    yaxis=dict(gridcolor="#e0e0e0"),
    legend=dict(x=0.6, y=0.95),
    font=dict(family="Helvetica Neue, Arial, sans-serif", color="#2b2b2b"),
)
chartF_html = pio.to_html(figF, full_html=False, include_plotlyjs=False)


# ══════════════════════════════════════════════════════════════════════════
# 5. PLOTLY CHARTS: SUPPLY CHAIN
# ══════════════════════════════════════════════════════════════════════════

print("Building SC charts...")

# ── SC1: Top 15 Producers ────────────────────────────────────────────────

top15 = cu_prod.head(15).copy()
top15["label"] = top15["FACILITY_NAME"].str.replace("División ", "", regex=False)
top15["pct"] = (top15["COCHILCO_CU_2024_KMT"] / cu_total * 100).round(1)

figSC1 = go.Figure()
figSC1.add_trace(go.Bar(
    y=top15["label"][::-1], x=top15["COCHILCO_CU_2024_KMT"][::-1],
    orientation="h", marker_color=PAL["active"],
    text=[f"{v:,.0f} kMT ({p:.1f}%)" for v, p in
          zip(top15["COCHILCO_CU_2024_KMT"][::-1], top15["pct"][::-1])],
    textposition="outside", textfont=dict(size=10),
))
figSC1.update_layout(
    title=dict(text=f"Top 15 Copper Producers (2024)<br><sub>{len(cu_prod)} active operations, {cu_total:,.0f} kMT matched</sub>",
               font=dict(size=14)),
    xaxis_title="Production (kMT)", height=480,
    margin=dict(l=140, r=80, t=80, b=50),
    plot_bgcolor="#f8f9fa", paper_bgcolor="white",
    xaxis=dict(gridcolor="#e0e0e0"),
    font=dict(family="Helvetica Neue, Arial, sans-serif", color="#2b2b2b"),
)
chartSC1_html = pio.to_html(figSC1, full_html=False, include_plotlyjs=False)

# ── SC2: Cumulative Curve ────────────────────────────────────────────────

figSC2 = go.Figure()
figSC2.add_trace(go.Scatter(
    x=cu_prod["rank"], y=cu_prod["cumulative_pct"],
    mode="lines", fill="tozeroy",
    line=dict(color=PAL["active"], width=2.5),
    fillcolor="rgba(44,62,107,0.1)", name="Cumulative %",
))
for threshold in [50, 80, 95]:
    idx = np.searchsorted(cu_prod["cumulative_pct"].values, threshold)
    if idx < len(cu_prod):
        figSC2.add_annotation(
            x=cu_prod["rank"].iloc[idx], y=threshold,
            text=f"{threshold}% from top {cu_prod['rank'].iloc[idx]} mines",
            showarrow=True, arrowhead=2, arrowcolor="#c0392b",
            font=dict(size=10, color="#c0392b"),
        )
        figSC2.add_trace(go.Scatter(
            x=[cu_prod["rank"].iloc[idx]], y=[threshold],
            mode="markers", marker=dict(size=8, color="#c0392b"), showlegend=False,
        ))
figSC2.update_layout(
    title=dict(text="Copper Production Concentration Curve", font=dict(size=14)),
    xaxis_title="Mine rank", yaxis_title="Cumulative % of total",
    height=400, margin=dict(l=60, r=40, t=60, b=50),
    plot_bgcolor="#f8f9fa", paper_bgcolor="white",
    xaxis=dict(gridcolor="#e0e0e0"), yaxis=dict(gridcolor="#e0e0e0", range=[0, 105]),
    font=dict(family="Helvetica Neue, Arial, sans-serif", color="#2b2b2b"),
)
chartSC2_html = pio.to_html(figSC2, full_html=False, include_plotlyjs=False)

# ── SC3: Edge Composition ────────────────────────────────────────────────

edge_counts = edges["EDGE_TYPE"].value_counts()
edge_labels = {
    "mine_to_plant": "Mine to Plant", "port_to_country": "Port to Country",
    "sxew_to_port": "SX-EW to Port", "concentrate_to_port": "Concentrate to Port",
    "smelter_to_port": "Smelter to Port", "concentrate_to_smelter": "Concentrate to Smelter",
}
edge_color_map = {
    "mine_to_plant": PAL["active"], "port_to_country": PAL["processing"],
    "sxew_to_port": PAL["prospect"], "concentrate_to_port": "#6c3483",
    "smelter_to_port": "#e87461", "concentrate_to_smelter": "#c0392b",
}

figSC3 = go.Figure()
figSC3.add_trace(go.Bar(
    y=[edge_labels.get(et, et) for et in edge_counts.index],
    x=edge_counts.values, orientation="h",
    marker_color=[edge_color_map.get(et, "#7f8c8d") for et in edge_counts.index],
    text=[f"{v} ({v/len(edges)*100:.1f}%)" for v in edge_counts.values],
    textposition="outside", textfont=dict(size=10),
))
figSC3.update_layout(
    title=dict(text=f"Supply Chain Edge Types<br><sub>{len(edges)} total edges</sub>",
               font=dict(size=14)),
    xaxis_title="Number of edges", height=380,
    margin=dict(l=180, r=80, t=80, b=50),
    plot_bgcolor="#f8f9fa", paper_bgcolor="white",
    xaxis=dict(gridcolor="#e0e0e0"),
    font=dict(family="Helvetica Neue, Arial, sans-serif", color="#2b2b2b"),
)
chartSC3_html = pio.to_html(figSC3, full_html=False, include_plotlyjs=False)

# ── SC4: Export Destinations ─────────────────────────────────────────────

cu_exports = exports[exports["COMMODITIES"] == "Copper"].copy()
if len(cu_exports) > 0 and "DESTINATION_TOTAL" in cu_exports.columns:
    cu_by_country = cu_exports.drop_duplicates(
        subset=["TO_NAME", "PRODUCT_FORM"]
    ).groupby("TO_NAME")["DESTINATION_TOTAL"].sum().sort_values(ascending=False)
    top10_exp = cu_by_country.head(10)
    total_exp = cu_by_country.sum()

    figSC4 = go.Figure()
    figSC4.add_trace(go.Bar(
        y=top10_exp.index[::-1], x=top10_exp.values[::-1],
        orientation="h", marker_color=PAL["active"],
        text=[f"{v:,.0f} kMT ({v/total_exp*100:.1f}%)" for v in top10_exp.values[::-1]],
        textposition="outside", textfont=dict(size=10),
    ))
    figSC4.update_layout(
        title=dict(text="Top 10 Copper Export Destinations (2024)", font=dict(size=14)),
        xaxis_title="Volume (kMT)", height=420,
        margin=dict(l=120, r=100, t=60, b=50),
        plot_bgcolor="#f8f9fa", paper_bgcolor="white",
        xaxis=dict(gridcolor="#e0e0e0"),
        font=dict(family="Helvetica Neue, Arial, sans-serif", color="#2b2b2b"),
    )
    chartSC4_html = pio.to_html(figSC4, full_html=False, include_plotlyjs=False)
else:
    chartSC4_html = "<p>Copper export data not available.</p>"


# ══════════════════════════════════════════════════════════════════════════
# 6. FOLIUM MAP
# ══════════════════════════════════════════════════════════════════════════

print("Building map...")

m = folium.Map(location=[-26.0, -70.0], zoom_start=5,
               tiles="cartodbpositron", width="100%", height="100%")

fg_active   = folium.FeatureGroup(name="Active Mines", show=True)
fg_idle     = folium.FeatureGroup(name="Idle Mines", show=True)
fg_prospect = folium.FeatureGroup(name="Prospects / Projects", show=True)
fg_process  = folium.FeatureGroup(name="Processing Facilities", show=True)
fg_ports    = folium.FeatureGroup(name="Ports", show=True)
fg_edges    = folium.FeatureGroup(name="Supply Chain Links", show=False)

def get_popup_html(row, status):
    cu_val = row.get("COCHILCO_CU_2024_KMT", None)
    op = row.get("OPERATOR_NAME", "")
    comm = row.get("PRIMARY_COMMODITY", "")
    lines = [f"<b>{row['FACILITY_NAME']}</b>"]
    lines.append(f"<span style='color:{PAL.get(status, '#666')};font-weight:600'>{row['FACILITY_TYPE']}</span>")
    if pd.notna(comm) and comm:
        lines.append(f"Primary: {comm}")
    if pd.notna(cu_val) and cu_val > 0:
        lines.append(f"Cu 2024: {cu_val:,.1f} kMT")
    if pd.notna(op) and op:
        lines.append(f"Operator: {op}")
    if status == "prospect":
        for suffix in ["_Resource", "_Reserve"]:
            rc = [c for c in row.index if c.endswith(suffix) and pd.notna(row[c])]
            for c in rc[:3]:
                commodity = c.rsplit("_", 1)[0]
                lines.append(f"{commodity} {suffix[1:]}: {row[c]:,.1f}")
    return "<br>".join(lines)

for _, row in active_mines[active_mines["LATITUD"].notna()].iterrows():
    cu = row.get("COCHILCO_CU_2024_KMT", 0)
    cu = cu if pd.notna(cu) else 0
    radius = max(4, min(20, cu / 60)) if cu > 0 else 3.5
    folium.CircleMarker(
        location=[row["LATITUD"], row["LONGITUD"]], radius=radius,
        color=PAL["active"], fill=True, fill_color=PAL["active"],
        fill_opacity=0.75, weight=1.0, opacity=0.9,
        popup=folium.Popup(get_popup_html(row, "active"), max_width=280),
    ).add_to(fg_active)

for _, row in idle_mines[idle_mines["LATITUD"].notna()].iterrows():
    folium.CircleMarker(
        location=[row["LATITUD"], row["LONGITUD"]], radius=3.5,
        color=PAL["idle"], fill=True, fill_color=PAL["idle"],
        fill_opacity=0.6, weight=0.8, opacity=0.7,
        popup=folium.Popup(get_popup_html(row, "idle"), max_width=280),
    ).add_to(fg_idle)

for _, row in prospects[prospects["LATITUD"].notna()].iterrows():
    folium.CircleMarker(
        location=[row["LATITUD"], row["LONGITUD"]], radius=5,
        color=PAL["prospect"], fill=True, fill_color=PAL["prospect"],
        fill_opacity=0.8, weight=1.5, opacity=0.9, dash_array="4 2",
        popup=folium.Popup(get_popup_html(row, "prospect"), max_width=300),
    ).add_to(fg_prospect)

PROC_COLORS = {
    "concentration": "#6c3483", "sx_ew": "#d4940a",
    "smelting": "#c0392b", "refining": "#e87461",
    "processing": "#1b7a6e", "other": "#7f8c8d",
}
for _, row in processing[processing["LATITUD"].notna()].iterrows():
    stage = row.get("CHAIN_STAGE_V2", "processing")
    color = PROC_COLORS.get(stage, "#7f8c8d")
    folium.CircleMarker(
        location=[row["LATITUD"], row["LONGITUD"]], radius=4,
        color=color, fill=True, fill_color=color,
        fill_opacity=0.7, weight=0.8, opacity=0.85,
        popup=folium.Popup(get_popup_html(row, "processing"), max_width=280),
    ).add_to(fg_process)

for _, p in ports.iterrows():
    folium.Marker(
        location=[p["lat"], p["lon"]],
        icon=folium.Icon(color="darkblue", icon="anchor", prefix="fa"),
        popup=folium.Popup(
            f"<b>{p['name']}</b><br>Region: {p['region']}<br>Products: {p['products']}",
            max_width=250),
    ).add_to(fg_ports)

EDGE_COLORS_MAP = {
    "mine_to_plant": "#2c3e6b", "concentrate_to_port": "#6c3483",
    "concentrate_to_smelter": "#c0392b", "sxew_to_port": "#d4940a",
    "smelter_to_port": "#e87461",
}
for _, e in edges[edges["EDGE_TYPE"].isin(list(EDGE_COLORS_MAP.keys()))].iterrows():
    if pd.isna(e.get("FROM_LAT")) or pd.isna(e.get("TO_LAT")):
        continue
    color = EDGE_COLORS_MAP.get(e["EDGE_TYPE"], "#999999")
    weight = 1.2 if e["EDGE_TYPE"] == "mine_to_plant" else 2.0
    opacity = 0.3 if e["EDGE_TYPE"] == "mine_to_plant" else 0.55
    folium.PolyLine(
        locations=[[e["FROM_LAT"], e["FROM_LON"]], [e["TO_LAT"], e["TO_LON"]]],
        color=color, weight=weight, opacity=opacity,
        popup=f"{e['EDGE_TYPE']}: {e['FROM_NAME']} -> {e['TO_NAME']}",
    ).add_to(fg_edges)

for fg in [fg_active, fg_idle, fg_prospect, fg_process, fg_ports, fg_edges]:
    fg.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

legend_html = """
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
     background:white; padding:14px 18px; border-radius:8px;
     border:1px solid #ccc; font-size:11px; line-height:1.7; max-width:220px;
     box-shadow: 0 2px 8px rgba(0,0,0,0.12);">
<b style="font-size:12px;">Site Status</b><br>
<span style="color:#2c3e6b;">&#11044;</span> Active Mine &nbsp;
<span style="color:#a0a8b4;">&#11044;</span> Idle Mine<br>
<span style="color:#d4940a;">&#11044;</span> Prospect / Project<br>
<b style="font-size:12px; margin-top:6px; display:block;">Processing</b>
<span style="color:#6c3483;">&#11044;</span> Concentrator &nbsp;
<span style="color:#d4940a;">&#11044;</span> SX-EW<br>
<span style="color:#c0392b;">&#11044;</span> Smelter &nbsp;
<span style="color:#e87461;">&#11044;</span> Refinery<br>
<span style="color:#1b7a6e;">&#11044;</span> Other Processing<br>
<b style="font-size:12px; margin-top:6px; display:block;">Edges</b>
<span style="color:#2c3e6b;">&#9135;</span> Mine-Plant &nbsp;
<span style="color:#6c3483;">&#9135;</span> Conc-Port<br>
<span style="color:#d4940a;">&#9135;</span> SX-EW-Port &nbsp;
<span style="color:#e87461;">&#9135;</span> Smelter-Port<br>
<i style="color:#888; font-size:9px;">Circle size = Cu production (active only)</i>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

map_path = os.path.join(OUT_DIR, "Chile_Map_Layered.html")
m.save(map_path)
print(f"Map saved: {map_path}")


# ══════════════════════════════════════════════════════════════════════════
# 7. SOURCE TABLE HTML
# ══════════════════════════════════════════════════════════════════════════

source_table_rows = ""
for _, r in source_df.iterrows():
    source_table_rows += f"""<tr>
      <td><code>{r['file']}</code></td>
      <td style="text-align:right">{r['records']:,}</td>
      <td style="text-align:right">{r['columns']}</td>
      <td style="text-align:right">{r['size_kb']:,.0f} KB</td>
    </tr>"""


# ══════════════════════════════════════════════════════════════════════════
# 8. ASSEMBLE FINAL HTML
# ══════════════════════════════════════════════════════════════════════════

print("Assembling dashboard...")

html_out = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Chile Mineral Supply Chain Dashboard</title>
<script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
<style>
  @import url('https://fonts.googleapis.com/css2?family=Source+Sans+3:wght@300;400;600;700&display=swap');
  * {{ margin: 0; padding: 0; box-sizing: border-box; }}
  body {{
    font-family: 'Source Sans 3', 'Helvetica Neue', Arial, sans-serif;
    background: #f0f2f5; color: #2b2b2b; line-height: 1.5;
  }}
  .header {{
    background: linear-gradient(135deg, #1a2744 0%, #2c3e6b 60%, #3d5a8a 100%);
    color: white; padding: 36px 48px; border-bottom: 3px solid #d4940a;
  }}
  .header h1 {{ font-size: 28px; font-weight: 700; letter-spacing: -0.5px; margin-bottom: 6px; }}
  .header p {{ font-size: 14px; font-weight: 300; color: rgba(255,255,255,0.75); max-width: 800px; }}
  .content {{ max-width: 1400px; margin: 0 auto; padding: 28px 32px; }}
  .section {{ margin-bottom: 40px; }}
  .section-title {{
    font-size: 20px; font-weight: 700; color: #1a2744;
    margin-bottom: 14px; padding-bottom: 6px;
    border-bottom: 2px solid #d4940a; display: inline-block;
  }}
  .section-desc {{
    font-size: 13px; color: #555; margin-bottom: 18px;
    max-width: 900px; line-height: 1.65;
  }}
  .card {{
    background: white; border-radius: 8px; padding: 24px 28px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08); border: 1px solid #e0e0e0;
    margin-bottom: 20px;
  }}
  .card h3 {{ font-size: 15px; font-weight: 700; color: #1a2744; margin-bottom: 10px; }}
  .card p {{ font-size: 13px; color: #555; margin-bottom: 8px; line-height: 1.6; }}
  table.src-table {{
    width: 100%; border-collapse: collapse; font-size: 13px; margin-top: 12px;
  }}
  table.src-table th {{
    text-align: left; padding: 9px 12px; background: #f0f2f5;
    border-bottom: 2px solid #d0d0d0; font-weight: 600; color: #333;
  }}
  table.src-table td {{ padding: 8px 12px; border-bottom: 1px solid #eee; color: #444; }}
  table.src-table tr:hover {{ background: #f8f9fa; }}
  table.src-table code {{
    background: #eef1f5; padding: 2px 6px; border-radius: 3px;
    font-size: 12px; color: #2c3e6b;
  }}
  .stats-grid {{
    display: grid; grid-template-columns: repeat(auto-fit, minmax(150px, 1fr));
    gap: 14px; margin: 18px 0;
  }}
  .mini-stat {{
    background: #f8f9fa; border-radius: 6px; padding: 14px 16px;
    border: 1px solid #e8e8e8; text-align: center;
  }}
  .mini-stat .val {{ font-size: 24px; font-weight: 700; line-height: 1.1; }}
  .mini-stat .lbl {{
    font-size: 10px; text-transform: uppercase; letter-spacing: 0.6px;
    color: #888; margin-top: 4px;
  }}
  .map-container {{
    background: white; border-radius: 8px; overflow: hidden;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08); border: 1px solid #e0e0e0;
  }}
  .map-container iframe {{ width: 100%; height: 620px; border: none; }}
  .chart-grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 24px; margin-bottom: 24px; }}
  .chart-card {{
    background: white; border-radius: 8px; padding: 16px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08); border: 1px solid #e0e0e0;
  }}
  .chart-card.full-width {{ grid-column: 1 / -1; }}
  .footer {{ text-align: center; padding: 20px; font-size: 11px; color: #999; margin-top: 20px; }}
  @media (max-width: 900px) {{
    .chart-grid {{ grid-template-columns: 1fr; }}
    .stats-grid {{ grid-template-columns: repeat(2, 1fr); }}
    .header {{ padding: 24px; }}
    .content {{ padding: 16px; }}
  }}
</style>
</head>
<body>

<div class="header">
  <h1>Chile Mineral Supply Chain</h1>
  <p>Facility inventory, industry context, data quality assessment, and supply chain analysis.
     Chile produces approximately 27% of global copper supply, with mining contributing over 10%
     of GDP. Active operations, idle capacity, and exploration-stage prospects are classified
     separately. Only active mines with matched production data feed into supply chain
     construction and export routing.</p>
</div>

<div class="content">

  <!-- ════════════════════════════════════════════════════════════ -->
  <!-- SECTION 1: DATA SOURCES                                     -->
  <!-- ════════════════════════════════════════════════════════════ -->

  <div class="section">
    <div class="section-title">1. Data Sources</div>

    <div class="card">
      <h3>Primary Sources</h3>
      <p><b>USGS Mineral Resources Data System (MRDS).</b> Provides the base facility inventory
         for Chile: mine locations, operator names, commodity classifications, and resource/reserve
         estimates. Each record corresponds to a distinct mineral occurrence or facility, typed as
         active mine, idle mine, prospect/project, or processing plant.</p>
      <p><b>Sernageomin (Chilean National Geology and Mining Service).</b> Supplements USGS records
         with domestic facility registrations, geocoordinates (latitude/longitude), and regional
         administrative codes (Region column). The combined USGS-Sernageomin inventory is the
         spatial backbone of the supply chain graph.</p>
      <p><b>COCHILCO Anuario Estadisticas del Cobre 2005-2024.</b> Copper production volumes by
         mine (Section A), molybdenum and lithium output, and export breakdowns by product form
         (concentrate, cathode, blister) and destination country (Sections C/D). 2024 production
         figures are matched to inventory records by name to assign volumes.</p>
      <p><b>Chilean Aduanas (Customs) Salidas FOB.</b> Port-level export shares used for
         mine-to-port routing weights, with product-specific breakdowns by departure port.</p>
      <p><b>UN Comtrade HS6 (2024).</b> Cross-validation of aggregate export volumes and FOB
         values against COCHILCO and Aduanas figures at the commodity level.</p>
      <p><b>Global Business Reports, Chile Mining 2021.</b> Industry report based on over 70
         interviews with mining companies, service providers, equipment suppliers and government
         officials. Used as a secondary reference for structural context on production trends,
         investment pipelines, exploration dynamics, water/energy challenges, and the regulatory
         environment. The report's 2020-2021 figures provide historical baseline context for
         the 2024 production data used in this dashboard.</p>
    </div>

    <div class="card">
      <h3>Intermediate Data Files</h3>
      <p>The main analysis notebook processes these raw sources into five CSV files that serve
         as inputs to this dashboard. The inventory file consolidates facility attributes, matched
         production, and resource/reserve data. The edges file encodes the directed supply chain
         graph (mine to concentrator, to smelter, to port, to destination country).</p>
      <table class="src-table">
        <thead>
          <tr><th>File</th><th style="text-align:right">Records</th><th style="text-align:right">Columns</th><th style="text-align:right">Size</th></tr>
        </thead>
        <tbody>{source_table_rows}</tbody>
      </table>
    </div>
  </div>

  <!-- ════════════════════════════════════════════════════════════ -->
  <!-- SECTION 2: INDUSTRY CONTEXT                                 -->
  <!-- ════════════════════════════════════════════════════════════ -->

  <div class="section">
    <div class="section-title">2. Chile's Mining Sector in Context</div>

    <div class="card">
      <h3>Macroeconomic Significance</h3>
      <p>Chile is the world's largest copper producer, accounting for approximately 27% of
         global supply. Mining contributes over 10% of national GDP, exceeds 50% of total
         exports, and represents around 25% of all investment received by the country. The
         sector's weight in the economy leaves Chile exposed to copper price fluctuations and
         demand shifts from China, its principal trading partner for the metal. Chile was the
         first Latin American country to join the OECD, and from 1980 to 2019, GDP per capita
         quintupled, a trajectory closely tied to mining-led growth.</p>
      <p>Between 2020 and 2029, COCHILCO projected a mining investment pipeline of US$74
         billion across 49 projects, predominantly copper-related. Of these, 68% are brownfield
         expansions of existing operations, 34% are in the execution stage, and 64% remain at
         feasibility. This brownfield orientation reflects the maturity of Chile's deposit base and
         the declining ore grades at its oldest operations.</p>
    </div>

    <div class="card">
      <h3>Production Trends and Structural Challenges</h3>
      <p>Chilean copper output stabilized near 5.7 to 5.8 million metric tonnes per year over the
         past decade, after a period of consistent growth that ended around 2013. The stabilization
         is driven by declining ore grades at legacy operations such as Escondida (BHP),
         Chuquicamata (Codelco), and Andina (Codelco), which have been in production for
         decades. Total factor productivity in Chilean copper mining has decreased at an average
         rate of 4.7% per year since 1993, according to the OECD, as hauling distances increase,
         rock hardness rises, and mineral grades fall.</p>
      <p>To offset these trends, major operators are undertaking large capital programs. Codelco
         has committed approximately US$40 billion over a decade to extend the lives of its core
         assets, with projects including the Chuquicamata underground conversion, the El Teniente
         Recursos Norte expansion, and Rajo Inca at the Salvador division. BHP's Spence expansion,
         Teck Resources' Quebrada Blanca Phase 2, and Antofagasta Plc's Los Pelambres expansion
         are additional projects that entered commissioning in recent years. The production
         profile is also shifting from hydrometallurgical output (cathodes via SX-EW) toward
         concentrate production, which COCHILCO projects will represent 54.9% of total copper
         output by 2031.</p>
    </div>

    <div class="card">
      <h3>Exploration and Future Pipeline</h3>
      <p>Chile registered an exploration budget of US$458 million in 2020, the fourth highest
         globally and equivalent to 5.5% of the world's nonferrous exploration spending (S&amp;P
         Global/COCHILCO). However, the composition is heavily skewed: major companies account
         for roughly 85% of exploration expenditure, focused predominantly on brownfield resource
         updates at existing deposits. Of the 101 companies with exploration projects in Chile,
         75 are junior companies, mostly headquartered in Canada or Australia, but they represent
         a comparatively smaller share of total budgets than in other mining jurisdictions.</p>
      <p>The concentration of exploration spending on brownfield work, combined with limited
         greenfield activity, raises concerns about the long-term replacement of depleting reserves.
         Much of northern Chile's prospective ground is held under concessions by major operators
         who may not be actively exploring, creating a barrier for junior entrants. In terms of
         commodities, gold represented 28% of exploration projects in Chile and 48% of holes
         drilled in 2020, indicating growing interest beyond copper. Chile holds approximately 52%
         of the world's lithium reserves (Ministry of Mining), though exploitation remains limited
         by lithium's classification as a strategic non-concessionary resource since 1979.</p>
    </div>

    <div class="card">
      <h3>Water, Energy, and Sustainability</h3>
      <p>Chile is experiencing its worst drought in six decades, with the arid northern mining
         regions facing acute freshwater scarcity. The mining industry consumes enough water
         annually to supply approximately 75% of Chile's population, with net freshwater use
         in copper mining averaging 0.5 to 0.7 cubic meters per tonne of ore processed. As a
         response, the industry is shifting toward seawater desalination and water recycling.
         By 2031, COCHILCO projects that 47% of the water used in mining will come from the
         sea, with 27 desalination plants operating by 2028. BHP's Escondida Water Supply plant
         is currently the largest desalination facility in the Americas.</p>
      <p>On the energy side, electricity demand for copper mining is forecast to grow by 34%
         over the decade, from 25 TWh in 2020 to 33.4 TWh by 2031. Mining companies have
         responded by transitioning to renewable energy contracts: COCHILCO data indicates
         renewable sources will supply approximately 49% of the copper industry's power
         requirements by 2023. Anglo American's Chilean operations run on 100% renewable
         electricity, and Antofagasta Plc completed the transition of all mining operations to
         renewables by 2022. Green hydrogen is being explored as a longer-term decarbonization
         pathway, particularly for heavy haulage.</p>
    </div>
  </div>

  <!-- ════════════════════════════════════════════════════════════ -->
  <!-- SECTION 3: GENERAL DATA OVERVIEW                            -->
  <!-- ════════════════════════════════════════════════════════════ -->

  <div class="section">
    <div class="section-title">3. Inventory Data Overview</div>
    <p class="section-desc">
      The merged inventory contains {len(inv):,} facility records covering the full spectrum of
      Chile's mineral sector, from active producing mines to early-stage exploration prospects.
      Each record is classified into one of four status categories based on its FACILITY_TYPE field.
      "Active" covers Mine (active) and Mine (USGS) records; "Idle" covers Mine (idle); "Prospect"
      covers Prospect/Project; everything else (concentrators, SX-EW plants, smelters, refineries,
      pellet/grinding/steel plants) falls under "Processing". This classification determines which
      sites enter production totals and supply chain construction: only active mines with non-zero
      COCHILCO 2024 production data are included. Idle mines and prospects appear in the
      inventory and on the map for spatial reference but generate no supply chain edges or
      production figures. The charts below describe the composition of the inventory, its geographic
      and commodity coverage, and data completeness for resource and reserve estimates.
    </p>

    <div class="stats-grid">
      <div class="mini-stat">
        <div class="val" style="color:#2c3e6b">{len(active_mines)}</div>
        <div class="lbl">Active Mines</div>
      </div>
      <div class="mini-stat">
        <div class="val" style="color:#a0a8b4">{len(idle_mines)}</div>
        <div class="lbl">Idle Mines</div>
      </div>
      <div class="mini-stat">
        <div class="val" style="color:#d4940a">{len(prospects)}</div>
        <div class="lbl">Prospects</div>
      </div>
      <div class="mini-stat">
        <div class="val" style="color:#1b7a6e">{len(processing)}</div>
        <div class="lbl">Processing</div>
      </div>
      <div class="mini-stat">
        <div class="val">{n_commodities}</div>
        <div class="lbl">Commodities</div>
      </div>
      <div class="mini-stat">
        <div class="val">{n_regions}</div>
        <div class="lbl">Regions</div>
      </div>
      <div class="mini-stat">
        <div class="val">{pct_geocoded:.0f}%</div>
        <div class="lbl">Geocoded</div>
      </div>
      <div class="mini-stat">
        <div class="val">{n_res_fields}</div>
        <div class="lbl">Res/Rev Fields</div>
      </div>
    </div>

    <div class="chart-grid">
      <div class="chart-card">{chartA_html}</div>
      <div class="chart-card">{chartB_html}</div>
    </div>
    <div class="chart-grid">
      <div class="chart-card full-width">{chartC_html}</div>
    </div>
    <div class="chart-grid">
      <div class="chart-card">{chartE_html}</div>
      <div class="chart-card">{chartD_html}</div>
    </div>
    <div class="chart-grid">
      <div class="chart-card full-width">{chartF_html}</div>
    </div>
  </div>

  <!-- ════════════════════════════════════════════════════════════ -->
  <!-- SECTION 4: SUPPLY CHAIN ANALYSIS                            -->
  <!-- ════════════════════════════════════════════════════════════ -->

  <div class="section">
    <div class="section-title">4. Supply Chain Analysis</div>
    <p class="section-desc">
      The supply chain graph connects {len(cu_prod)} active copper-producing mines to downstream
      processing facilities and {len(ports)} export ports via {len(edges):,} directed edges.
      Material flows are classified into {n_edge_types} link types covering the full
      mine-to-concentrator/SX-EW, concentrate-to-smelter, and product-to-port-to-country pipeline.
      Chile's copper supply chain includes six smelters (Chuquicamata, Potrerillos, Caletones,
      Altonorte, Paipote, and Chagres), the majority operated by Codelco or ENAMI, with
      Glencore's Altonorte and Anglo American's Chagres serving as custom smelters for
      third-party concentrate. The production matrix is shifting from SX-EW cathode output
      toward concentrate, reflecting both the depletion of oxide reserves at mature operations
      and the commissioning of new sulphide-processing capacity at projects such as Quebrada
      Blanca Phase 2, Mantoverde Sulphide Development, and the Spence concentrator.
      The map below shows all facility categories on separate toggleable layers; supply chain
      edges can be turned on via the layer control panel. Production totals reflect only
      COCHILCO-matched active mines ({cu_total:,.0f} kMT copper in 2024). Prospects and idle
      mines appear on the map for spatial context but generate no edges or production figures.
    </p>

    <div class="map-container">
      <iframe src="Chile_Map_Layered.html" loading="lazy"></iframe>
    </div>

    <div class="chart-grid" style="margin-top:24px">
      <div class="chart-card">{chartSC1_html}</div>
      <div class="chart-card">{chartSC2_html}</div>
    </div>
    <div class="chart-grid">
      <div class="chart-card">{chartSC3_html}</div>
      <div class="chart-card">{chartSC4_html}</div>
    </div>
  </div>

  <!-- CLASSIFICATION AND METHODOLOGY NOTES -->
  <div class="card">
    <h3>Classification Methodology</h3>
    <p><b>Active mines</b> cover FACILITY_TYPE "Mine (active)" and "Mine (USGS)". These are
       currently operating extraction sites with assigned production volumes. Chile's active
       copper mines range from world-class operations producing over 1 million mt/y (Escondida)
       to small-scale producers below 10,000 mt/y. The top 3 mines by production (Escondida,
       Collahuasi, El Teniente) are among the 10 largest copper mines globally.</p>
    <p><b>Idle mines</b> have FACILITY_TYPE "Mine (idle)" representing temporarily suspended
       operations. These sites may retain infrastructure and could resume production if economic
       conditions improve, but currently generate no output. They are retained in the inventory
       for spatial reference and future planning but excluded from supply chain edges and
       production matching.</p>
    <p><b>Prospects/Projects</b> have FACILITY_TYPE "Prospect/Project" and represent sites
       at exploration, feasibility, or permitting stages. No commercial production takes place.
       Chile's exploration pipeline includes both copper porphyry-type and iron-oxide-copper-gold
       (IOCG) deposits, with increasing interest in gold and lithium targets. Prospects appear on
       the map for forward-looking capacity assessment and to visualize the geographic
       distribution of future potential relative to active operations.</p>
    <p><b>Processing facilities</b> include concentrators, SX-EW plants, smelters, refineries,
       and other downstream units. Chile operates six copper smelters with a combined capacity
       handling a significant share of domestic concentrate. Processing facilities are classified
       by stage (concentration, smelting, refining, SX-EW) and connected to active mines via
       the supply chain edge table. The production matrix is shifting from hydrometallurgical
       copper (SX-EW cathodes) toward concentrate output as oxide reserves at mature
       operations are depleted.</p>
  </div>

</div>

<div class="footer">
  Chile Mineral Supply Chain Dashboard | Data as of 2024 | Generated with Python, Folium, Plotly
</div>

</body>
</html>"""

out_path = os.path.join(OUT_DIR, "Chile_Dashboard.html")
with open(out_path, "w", encoding="utf-8") as f:
    f.write(html_out)

print(f"\nDashboard saved: {out_path}")
print(f"File size: {os.path.getsize(out_path) / 1024 / 1024:.1f} MB")
print("Done.")

Loaded: 461 facilities, 1110 links, 1714 edges, 561 exports
Active: 74, Idle: 80, Prospects: 113, Processing: 194
Cu total (active matched): 5,197 kMT from 29 mines
Building overview charts...
Building SC charts...
Building map...


/var/folders/lk/thldsylx4nx779cggs7gnk900000gn/T/ipykernel_68928/989127752.py:287: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



Map saved: /Users/leoss/Desktop/GitHub/Capstone/Case studies/Chile/Outputs/Chile_Map_Layered.html
Assembling dashboard...

Dashboard saved: /Users/leoss/Desktop/GitHub/Capstone/Case studies/Chile/Outputs/Chile_Dashboard.html
File size: 0.1 MB
Done.
